In [ ]:
import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models 
# 데이터셋 클래스 (기존과 동일)
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask

# ERFNet 모델 정의
class DownsamplerBlock(nn.Module):
    def __init__(self, ninput, noutput):
        super().__init__()
        self.conv = nn.Conv2d(ninput, noutput - ninput, (3, 3), stride=2, padding=1, bias=False)
        self.pool = nn.MaxPool2d(2, stride=2)
        self.bn = nn.BatchNorm2d(noutput, eps=1e-3)

    def forward(self, input):
        output = torch.cat([self.conv(input), self.pool(input)], 1)
        output = self.bn(output)
        return F.relu(output)

class NonBottleneck1D(nn.Module):
    def __init__(self, chann, dropprob, dilated):
        super().__init__()
        # 패딩 값 조정
        self.conv3x1_1 = nn.Conv2d(chann, chann, (3, 1), stride=1, padding=(1, 0), bias=True)
        self.conv1x3_1 = nn.Conv2d(chann, chann, (1, 3), stride=1, padding=(0, 1), bias=True)
        self.bn1 = nn.BatchNorm2d(chann, eps=1e-03)
        
        # 팽창 컨볼루션 패딩 조정
        self.conv3x1_2 = nn.Conv2d(chann, chann, (3, 1), stride=1, 
                                  padding=(dilated, 0), dilation=(dilated, 1), bias=True)
        self.conv1x3_2 = nn.Conv2d(chann, chann, (1, 3), stride=1, 
                                  padding=(0, dilated), dilation=(1, dilated), bias=True)
        self.bn2 = nn.BatchNorm2d(chann, eps=1e-03)
        self.dropout = nn.Dropout2d(dropprob)

    def forward(self, input):
        output = self.conv3x1_1(input)
        output = F.relu(output)
        output = self.conv1x3_1(output)
        output = self.bn1(output)
        output = F.relu(output)

        output = self.conv3x1_2(output)
        output = F.relu(output)
        output = self.conv1x3_2(output)
        output = self.bn2(output)

        if self.dropout.p != 0:
            output = self.dropout(output)
            
        # Skip connection 전에 크기 확인 및 조정
        if output.shape != input.shape:
            output = F.interpolate(output, size=input.shape[2:], mode='bilinear', align_corners=True)
            
        return F.relu(output + input)

class ERFNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Encoder
        self.initial_block = DownsamplerBlock(3, 16)
        
        self.layers = nn.ModuleList()
        self.layers.append(DownsamplerBlock(16, 64))
        
        # 첫 번째 NonBottleneck 블록 (dilation=1)
        for _ in range(5):
            self.layers.append(NonBottleneck1D(64, 0.1, 1))
        
        self.layers.append(DownsamplerBlock(64, 128))
        
        # 팽창 컨볼루션 블록 (다양한 dilation rate)
        dilation_rates = [2, 4, 8, 16] * 2  # 2번 반복
        for rate in dilation_rates:
            self.layers.append(NonBottleneck1D(128, 0.1, rate))
        
        # Decoder
        self.decoder_layers = nn.ModuleList()
        
        # 첫 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(128, 64, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(64))
        self.decoder_layers.append(nn.ReLU())
        
        # 두 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(64, 16, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(16))
        self.decoder_layers.append(nn.ReLU())
        
        # 최종 출력 레이어
        self.output_conv = nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x = self.initial_block(x)
        for layer in self.layers:
            x = layer(x)
        
        # Decoder
        for layer in self.decoder_layers:
            x = layer(x)
        
        return self.output_conv(x)

# 시각화 함수 (기존과 동일)
def visualize_sample(image, mask, pred=None):
    plt.figure(figsize=(18, 6))
    
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img_np = np.clip(img_np, 0, 1)

    mask_np = mask.cpu().numpy().squeeze()

    overlay = img_np.copy()
    overlay[mask_np == 1] = [1, 0, 0]

    plt.subplot(1, 3, 1)
    plt.imshow(overlay)
    plt.title("Image + GT")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title("GT Mask")

    if pred is not None:
        plt.subplot(1, 3, 3)
        pred_resized = F.interpolate(
            pred.unsqueeze(0).float(),
            size=(img_np.shape[0], img_np.shape[1]),
            mode='bilinear',
            align_corners=True
        )
        pred_mask = torch.argmax(pred_resized, dim=1).squeeze().cpu().numpy()
        
        pred_overlay = img_np.copy()
        pred_overlay[pred_mask == 1] = [0, 1, 0]
        plt.imshow(pred_overlay)
        plt.title("Prediction")

    plt.tight_layout()
    plt.show()

# 평가 메트릭 (기존과 동일)
def calculate_metrics(outputs, targets):
    resized_targets = F.interpolate(targets.unsqueeze(1).float(), 
                                size=outputs.shape[2:], 
                                mode='nearest').squeeze(1).long()
    preds = torch.argmax(outputs, dim=1)
    
    intersection = (preds & resized_targets).float().sum()
    union = (preds | resized_targets).float().sum()
    iou = (intersection + 1e-6) / (union + 1e-6)
    
    tp = ((preds == 1) & (resized_targets == 1)).float().sum()
    fp = ((preds == 1) & (resized_targets == 0)).float().sum()
    fn = ((preds == 0) & (resized_targets == 1)).float().sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    return iou, precision, recall, f1

# 데이터 증강 (기존과 동일)
train_transform = A.Compose([
    A.Resize(608, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomShadow(shadow_roi=(0,0,1,0.5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.GridDropout(ratio=0.1, random_offset=True, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 데이터 로딩 (기존과 동일)
try:
    dataset = SDLaneDataset(
        image_dir='/kaggle/input/sdlane/SDLane/train/images',
        label_dir='/kaggle/input/sdlane/SDLane/train/labels',
        transform=train_transform
    )
    
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    val_dataset.dataset.transform = val_transform
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
    
    sample_img, sample_mask = dataset[0]
    visualize_sample(sample_img, sample_mask)

except Exception as e:
    print(f"Data loading error: {str(e)}")
    exit()

# 클래스 가중치 계산 (기존과 동일)
def calculate_class_weights(loader):
    lane_pixels = 0
    total_pixels = 0
    for _, masks in loader:
        lane_pixels += torch.sum(masks == 1).item()
        total_pixels += masks.numel()
    return torch.tensor([1.0, total_pixels/(2*lane_pixels)]).to(device)

# 모델 초기화 (ERFNet으로 변경)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ERFNet(num_classes=2).to(device)

# 가중치 계산
class_weights = calculate_class_weights(train_loader)
print(f"Class weights: {class_weights.cpu().numpy()}")

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
scaler = GradScaler()

# 학습 루프 (기존과 동일)
best_f1 = 0.0
early_stopping_patience = 5
patience_counter = 0

for epoch in range(100):
    model.train()
    train_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, F.interpolate(masks.unsqueeze(1).float(), 
                                         size=outputs.shape[2:], 
                                         mode='nearest').squeeze(1).long())
            iou, prec, rec, f1 = calculate_metrics(outputs, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_metrics['iou'] += iou.item() * images.size(0)
        train_metrics['precision'] += prec.item() * images.size(0)
        train_metrics['recall'] += rec.item() * images.size(0)
        train_metrics['f1'] += f1.item() * images.size(0)
    
    model.eval()
    val_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            
            outputs = model(images)
            iou, prec, rec, f1 = calculate_metrics(outputs, masks)
            
            val_metrics['iou'] += iou.item() * images.size(0)
            val_metrics['precision'] += prec.item() * images.size(0)
            val_metrics['recall'] += rec.item() * images.size(0)
            val_metrics['f1'] += f1.item() * images.size(0)
    
    train_metrics = {k: v/len(train_dataset) for k,v in train_metrics.items()}
    val_metrics = {k: v/len(val_dataset) for k,v in val_metrics.items()}
    
    scheduler.step(val_metrics['iou'])
    
    print(f"\nEpoch {epoch+1:03d}")
    print(f"Train | IoU: {train_metrics['iou']:.4f} | Prec: {train_metrics['precision']:.4f} | Rec: {train_metrics['recall']:.4f} | F1: {train_metrics['f1']:.4f}")
    print(f"Val   | IoU: {val_metrics['iou']:.4f} | Prec: {val_metrics['precision']:.4f} | Rec: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")
    
    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Model improved (F1: {best_f1:.4f}) - Saved")
        
        model.eval()
        with torch.no_grad():
            val_images, val_masks = next(iter(val_loader))
            val_images, val_masks = val_images.to(device), val_masks.to(device)
            val_outputs = model(val_images)
            
            for i in range(min(3, len(val_images))):
                visualize_sample(val_images[i], val_masks[i], val_outputs[i])
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{early_stopping_patience})")
    
    if patience_counter >= early_stopping_patience:
        print("\nEarly stopping triggered!")
        break

# 최종 모델 평가
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

In [ ]:
import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
from torchvision.models import vgg16

# 데이터셋 클래스 (LaneNet용으로 수정)
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        # 이미지 로드
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # JSON 주석 로드
        with open(label_path, 'r') as f:
            label_data = json.load(f)
        
        # 마스크 생성
        binary_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        instance_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane_idx, lane in enumerate(lanes, start=1):
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(binary_mask, [points], isClosed=False, color=1, thickness=4)
                cv2.polylines(instance_mask, [points], isClosed=False, color=lane_idx, thickness=4)
        
        # 후처리
        kernel = np.ones((3, 3), np.uint8)
        binary_mask = cv2.dilate(binary_mask, np.ones((3,3)), iterations=1)
        instance_mask = cv2.dilate(instance_mask, np.ones((3,3)), iterations=1)
        # 변환 적용 (numpy 배열로 전달)
        if self.transform:
            # Albumentations에 전달하기 전에 반드시 numpy 배열인지 확인
            transformed = self.transform(
                image=image.astype(np.uint8),  # 이미지는 uint8 형식으로
                masks=[
                    binary_mask.astype(np.uint8),  # 바이너리 마스크
                    instance_mask.astype(np.uint8)  # 인스턴스 마스크
                ]
            )
            image = transformed['image']
            binary_mask = transformed['masks'][0].long()
            instance_mask = transformed['masks'][1].long()
        else:
            # 변환 없을 경우 torch.Tensor로 변환
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            binary_mask = torch.from_numpy(binary_mask).long()
            instance_mask = torch.from_numpy(instance_mask).long()
        
        return image, {'binary': binary_mask, 'instance': instance_mask}
# LaneNet 모델 구현
ne Network (VGG16 기반)
        backbone = vgg16(weights='DEFAULT').features
        self.encoder1 = backbone[:4]    # /1
        self.encoder2 = backbone[4:9]   # /2
        self.encoder3 = backbone[9:16]  # /4
        self.encoder4 = backbone[16:23] # /8
        self.encoder5 = backbone[23:30] # /16

        # Binary Segmentation Decoder
        self.binary_decoder = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=3, p
# 데이터 증강 (멀티 마스크 지원)
train_transform = A.Compose([
    A.Resize(608, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomShadow(shadow_roi=(0,0,1,0.5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.GridDropout(ratio=0.1, random_offset=True, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={'masks': 'mask'})

val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={
    'mask0': 'mask',
    'mask1': 'mask'
})

# 데이터셋 및 데이터로더 초기화
try:
    dataset = SDLaneDataset(
        image_dir='/kaggle/input/sdlane/SDLane/train/images',
        label_dir='/kaggle/input/sdlane/SDLane/train/labels',
        transform=train_transform
    )
    
    # Train/Validation 분할
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    val_dataset.dataset.transform = val_transform
    
    # 데이터로더
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)
    
    # 샘플 확인
    sample_img, sample_masks = dataset[0]
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_masks['binary'].squeeze(), cmap='gray')
    plt.title("Binary Mask")
    plt.subplot(1, 2, 2)
    plt.imshow(sample_masks['instance'].squeeze())
    plt.title("Instance Mask")
    plt.show()

except Exception as e:
    print(f"Data loading error: {str(e)}")
    exit()

def calculate_metrics(pred, target):
    """Calculate IoU, Precision, Recall, F1 for binary segmentation"""
    pred = torch.argmax(pred, dim=1)  # Convert logits to class predictions
    target = target.long()
    
    # Flatten tensors
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    
    # Calculate TP, FP, FN
    tp = torch.sum((pred_flat == 1) & (target_flat == 1))
    fp = torch.sum((pred_flat == 1) & (target_flat == 0))
    fn = torch.sum((pred_flat == 0) & (target_flat == 1))
    
    # Calculate metrics
    eps = 1e-6  # to avoid division by zero
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    
    return iou, precision, recall, f1

def visualize_sample(image, mask, pred=None):
    plt.figure(figsize=(18, 6))
    
    # 이미지 정규화 해제
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img_np = np.clip(img_np, 0, 1)

    mask_np = mask.cpu().numpy().squeeze()

    # GT 시각화
    plt.subplot(1, 3, 1)
    overlay = img_np.copy()
    overlay[mask_np == 1] = [1, 0, 0]  # 빨간색으로 GT 표시
    plt.imshow(overlay)
    plt.title("Image + GT")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title("GT Mask")

    # 예측 결과 시각화
    if pred is not None:
        plt.subplot(1, 3, 3)
        pred_resized = F.interpolate(
            pred.unsqueeze(0).float(),
            size=(img_np.shape[0], img_np.shape[1]),
            mode='bilinear',
            align_corners=True
        )
        pred_mask = torch.argmax(pred_resized, dim=1).squeeze().cpu().numpy()
        
        pred_overlay = img_np.copy()
        pred_overlay[pred_mask == 1] = [0, 1, 0]  # 초록색으로 예측 표시
        plt.imshow(pred_overlay)
        plt.title("Prediction")

    plt.tight_layout()
    plt.show()
    
# 모델 초기화
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LaneNet(num_classes=2, embedding_dim=4).to(device)

# 손실 함수
criterion = LaneNetLoss()

# 옵티마이저
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
scaler = GradScaler()

# 학습 루프
best_f1 = 0.0
early_stopping_patience = 5
patience_counter = 0

for epoch in range(100):
    model.train()
    train_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    for images, targets in train_loader:
        images = images.to(device)
        binary_gt = targets['binary'].to(device)
        instance_gt = targets['instance'].to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            binary_pred, embedding_pred = model(images)
            loss = criterion((binary_pred, embedding_pred), (binary_gt, instance_gt))
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # 메트릭 계산 (이진 분할 결과만 평가)
        iou, prec, rec, f1 = calculate_metrics(binary_pred, binary_gt)
        train_metrics['iou'] += iou.item() * images.size(0)
        train_metrics['precision'] += prec.item() * images.size(0)
        train_metrics['recall'] += rec.item() * images.size(0)
        train_metrics['f1'] += f1.item() * images.size(0)
    
    # 검증 단계
    model.eval()
    val_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            binary_gt = targets['binary'].to(device)
            
            binary_pred, _ = model(images)  # 임베딩은 평가에 사용하지 않음
            iou, prec, rec, f1 = calculate_metrics(binary_pred, binary_gt)
            
            val_metrics['iou'] += iou.item() * images.size(0)
            val_metrics['precision'] += prec.item() * images.size(0)
            val_metrics['recall'] += rec.item() * images.size(0)
            val_metrics['f1'] += f1.item() * images.size(0)
    
    # 평균 계산 및 출력
    train_metrics = {k: v/len(train_dataset) for k,v in train_metrics.items()}
    val_metrics = {k: v/len(val_dataset) for k,v in val_metrics.items()}
    
    print(f"\nEpoch {epoch+1:03d}")
    print(f"Train | IoU: {train_metrics['iou']:.4f} | Prec: {train_metrics['precision']:.4f} | Rec: {train_metrics['recall']:.4f} | F1: {train_metrics['f1']:.4f}")
    print(f"Val   | IoU: {val_metrics['iou']:.4f} | Prec: {val_metrics['precision']:.4f} | Rec: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")
    
    # 모델 저장 (F1 기준)
    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_lanenet.pth')
        print(f"Model improved (F1: {best_f1:.4f}) - Saved")
        model.eval()
        with torch.no_grad():
            val_images, val_targets = next(iter(val_loader))
            val_images = val_images.to(device)
            val_binary_pred, _ = model(val_images)  # 임베딩은 시각화에 사용 안함
        
        # 상위 3개 샘플 시각화 (ERFNet과 동일한 방식)
            for i in range(min(3, len(val_images))):
                visualize_sample(
                    val_images[i],
                    val_targets['binary'][i],
                    val_binary_pred[i]
                )
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{early_stopping_patience})")
    
    if patience_counter >= early_stopping_patience:
        print("\nEarly stopping triggered!")
        break

model.load_state_dict(torch.load('best_lanenet.pth'))
model.eval()
num_test_samples = 3
for i in range(num_test_samples):
    test_img, test_masks = dataset[i]
    test_img = test_img.to(device).unsqueeze(0)
    
    with torch.no_grad():
        binary_pred, _ = model(test_img)  # 임베딩은 시각화에 사용 안함
        visualize_sample(
            test_img.squeeze(0),
            test_masks['binary'],
            binary_pred.squeeze(0)
        )

In [ ]:
pip install -U albumentations

In [ ]:
print("123")

In [ ]:
import os
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast


class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask
        

# ERFNet 모델 정의
class DownsamplerBlock(nn.Module):
    def __init__(self, ninput, noutput):
        super().__init__()
        self.conv = nn.Conv2d(ninput, noutput - ninput, (3, 3), stride=2, padding=1, bias=False)
        self.pool = nn.MaxPool2d(2, stride=2)
        self.bn = nn.BatchNorm2d(noutput, eps=1e-3)

    def forward(self, input):
        output = torch.cat([self.conv(input), self.pool(input)], 1)
        output = self.bn(output)
        return F.relu(output)

class NonBottleneck1D(nn.Module):
    def __init__(self, chann, dropprob, dilated):
        super().__init__()
        # 패딩 값 조정
        self.conv3x1_1 = nn.Conv2d(chann, chann, (3, 1), stride=1, padding=(1, 0), bias=True)
        self.conv1x3_1 = nn.Conv2d(chann, chann, (1, 3), stride=1, padding=(0, 1), bias=True)
        self.bn1 = nn.BatchNorm2d(chann, eps=1e-03)
        
        # 팽창 컨볼루션 패딩 조정
        self.conv3x1_2 = nn.Conv2d(chann, chann, (3, 1), stride=1, 
                                  padding=(dilated, 0), dilation=(dilated, 1), bias=True)
        self.conv1x3_2 = nn.Conv2d(chann, chann, (1, 3), stride=1, 
                                  padding=(0, dilated), dilation=(1, dilated), bias=True)
        self.bn2 = nn.BatchNorm2d(chann, eps=1e-03)
        self.dropout = nn.Dropout2d(dropprob)

    def forward(self, input):
        output = self.conv3x1_1(input)
        output = F.relu(output)
        output = self.conv1x3_1(output)
        output = self.bn1(output)
        output = F.relu(output)

        output = self.conv3x1_2(output)
        output = F.relu(output)
        output = self.conv1x3_2(output)
        output = self.bn2(output)

        if self.dropout.p != 0:
            output = self.dropout(output)
            
        # Skip connection 전에 크기 확인 및 조정
        if output.shape != input.shape:
            output = F.interpolate(output, size=input.shape[2:], mode='bilinear', align_corners=True)
            
        return F.relu(output + input)

class ERFNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Encoder
        self.initial_block = DownsamplerBlock(3, 16)
        
        self.layers = nn.ModuleList()
        self.layers.append(DownsamplerBlock(16, 64))
        
        # 첫 번째 NonBottleneck 블록 (dilation=1)
        for _ in range(5):
            self.layers.append(NonBottleneck1D(64, 0.1, 1))
        
        self.layers.append(DownsamplerBlock(64, 128))
        
        # 팽창 컨볼루션 블록 (다양한 dilation rate)
        dilation_rates = [2, 4, 8, 16] * 2  # 2번 반복
        for rate in dilation_rates:
            self.layers.append(NonBottleneck1D(128, 0.1, rate))
        
        # Decoder
        self.decoder_layers = nn.ModuleList()
        
        # 첫 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(128, 64, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(64))
        self.decoder_layers.append(nn.ReLU())
        
        # 두 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(64, 16, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(16))
        self.decoder_layers.append(nn.ReLU())
        
        # 최종 출력 레이어
        self.output_conv = nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x = self.initial_block(x)
        for layer in self.layers:
            x = layer(x)
        
        # Decoder
        for layer in self.decoder_layers:
            x = layer(x)
        
        return self.output_conv(x)

# 평가용 변환 (학습 시와 동일하게 유지)
val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 테스트 데이터셋 로드
test_dataset = SDLaneDataset(
    image_dir='/kaggle/input/sdlane/SDLane/test/images',
    label_dir='/kaggle/input/sdlane/SDLane/test/labels',
    transform=val_transform
)

# 배치 크기 줄여서 메모리 절약 (Kaggle 환경 고려)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)


In [ ]:

def evaluate_model(model, test_loader, device):
    model.eval()
    total_iou = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            targets = targets.to(device)
            
            outputs = model(images)
            
            # Resize predictions to match target dimensions
            if outputs.shape[2:] != targets.shape[1:]:
                outputs = F.interpolate(outputs, size=targets.shape[1:], mode='bilinear', align_corners=True)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Flatten predictions and targets
            preds_flat = preds.view(-1).cpu().numpy()
            targets_flat = targets.view(-1).cpu().numpy()
            
            # Calculate confusion matrix
            cm = confusion_matrix(targets_flat, preds_flat, labels=[0, 1])
            
            # Handle case where some classes might be missing
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
            else:
                # If one class is missing, fill with zeros
                if 0 not in np.unique(preds_flat) or 0 not in np.unique(targets_flat):
                    tn, fp, fn, tp = 0, 0, cm.sum(), 0
                else:
                    tn, fp, fn, tp = cm[0,0], 0, 0, cm[0,0]
            
            eps = 1e-6
            iou = tp / (tp + fp + fn + eps)
            precision = tp / (tp + fp + eps)
            recall = tp / (tp + fn + eps)
            f1 = 2 * (precision * recall) / (precision + recall + eps)
            
            total_iou += iou * images.size(0)
            total_precision += precision * images.size(0)
            total_recall += recall * images.size(0)
            total_f1 += f1 * images.size(0)
            
            all_preds.extend(preds_flat)
            all_targets.extend(targets_flat)
    
    avg_iou = total_iou / len(test_loader.dataset)
    avg_precision = total_precision / len(test_loader.dataset)
    avg_recall = total_recall / len(test_loader.dataset)
    avg_f1 = total_f1 / len(test_loader.dataset)
    
    return {
        'iou': avg_iou,
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': avg_f1,
        'confusion_matrix': confusion_matrix(all_targets, all_preds, labels=[0, 1])
    }
# 시각화 함수
def visualize_predictions(model, dataset, device, num_samples=3):
    plt.figure(figsize=(15, 5 * num_samples))
    
    for i in range(num_samples):
        image, gt_mask = dataset[i]
        image = image.unsqueeze(0).to(device)
        
        # 원본 이미지
        img_np = image.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)
        
        # GT 마스크
        gt_mask_np = gt_mask.cpu().numpy().squeeze()
        
        # 모델 예측
        with torch.no_grad():
            output = model(image)
            pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
        
        # 시각화
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(img_np)
        plt.title("Original Image")
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i*3 + 2)
        overlay = img_np.copy()
        overlay[gt_mask_np == 1] = [1, 0, 0]  # 빨간색으로 GT 표시
        plt.imshow(overlay)
        plt.title("Ground Truth")
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i*3 + 3)
        pred_overlay = img_np.copy()
        pred_overlay[pred == 1] = [0, 1, 0]  # 초록색으로 예측 표시
        plt.imshow(pred_overlay)
        plt.title("Prediction")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# 메인 평가 프로세스
def main():
    device = torch.device('cuda')
    
    # 모델 초기화 및 가중치 로드
    model = ERFNet(num_classes=2).to(device)
    model.load_state_dict(torch.load('/kaggle/input/erfnet/other/default/1/SDLane_ERFNet.pth'))
    model.eval()
    
    # 성능 평가
    print("Evaluating model on test set...")
    results = evaluate_model(model, test_loader, device)
    
    # 결과 출력
    print("\nTest Results:")
    print(f"IoU: {results['iou']:.4f}")
    print(f"Precision: {results['precision']:.4f}")
    print(f"Recall: {results['recall']:.4f}")
    print(f"F1 Score: {results['f1']:.4f}")
    
    # 혼동 행렬 시각화
    plt.figure(figsize=(8, 6))
    sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', xticklabels=['Background', 'Lane'], 
                yticklabels=['Background', 'Lane'])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    # 샘플 예측 시각화
    print("\nVisualizing sample predictions...")
    visualize_predictions(model, test_dataset, device, num_samples=3)

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models 

class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask
        
class SCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Backbone (ResNet34)
        backbone = models.resnet34(pretrained=True)
        self.encoder1 = nn.Sequential(*list(backbone.children())[:5])  # /2
        self.encoder2 = backbone.layer2  # /4
        self.encoder3 = backbone.layer3  # /8
        self.encoder4 = backbone.layer4  # /16

        # SCNN 모듈 (패딩 추가)
        self.scnn = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=(1,9), padding=(0,4)),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=(9,1), padding=(4,0)),
            nn.ReLU()
        )

        # 디코더 (크기 조정 보장)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, num_classes, kernel_size=1)  # 최종 출력
        )

    def forward(self, x):
        # 인코더
        e1 = self.encoder1(x)  # /2
        e2 = self.encoder2(e1) # /4
        e3 = self.encoder3(e2) # /8
        e4 = self.encoder4(e3) # /16
        
        # SCNN
        x = self.scnn(e4)
        
        # 디코더 (크기 자동 조정)
        x = self.decoder(x)
        return x
# 테스트를 위한 변환 (학습 시 사용한 val_transform과 동일하게 맞춤)
test_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 테스트 데이터셋 로드 (기존 코드와 동일)
test_dataset = SDLaneDataset(
    image_dir='/kaggle/input/sdlane/SDLane/train/images',  # 실제 테스트 시에는 test 디렉토리로 변경
    label_dir='/kaggle/input/sdlane/SDLane/train/labels',  # 실제 테스트 시에는 test 디렉토리로 변경
    transform=test_transform
)

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 저장된 모델 불러오기
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SCNN(num_classes=2).to(device)
model.load_state_dict(torch.load('/kaggle/input/scnn/other/default/1/SDLane_SCNN.pth'))
model.eval()

# 단일 이미지 테스트 함수
def test_single_image(model, image_path, transform):
    # 이미지 로드 및 전처리
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to load image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_h, original_w = image.shape[:2]  # 원본 이미지 크기 저장
    
    # 변환 적용
    if transform:
        transformed = transform(image=image, mask=np.zeros((image.shape[0], image.shape[1])))
        image_tensor = transformed['image'].unsqueeze(0).to(device)  # 배치 차원 추가
    else:
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float().unsqueeze(0).to(device)
    
    # 예측 수행
    with torch.no_grad():
        output = model(image_tensor)
        # 원본 이미지 크기로 리사이즈
        pred_mask = F.interpolate(output, size=(original_h, original_w), 
                                mode='bilinear', align_corners=True)
        pred_mask = torch.argmax(pred_mask, dim=1).squeeze().cpu().numpy()
    
    # 원본 이미지 복원
    img_np = image_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # 정규화 역변환
    img_np = np.clip(img_np, 0, 1)
    
    # 예측 결과 오버레이 (초록색)
    overlay = img_np.copy()
    overlay[pred_mask == 1] = [0, 1, 0]  # 차선 영역만 초록색
    
    # 시각화
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.title("Original Image")
    
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("Predicted Lane Overlay")
    
    plt.tight_layout()
    plt.show()
    
    return pred_mask
# 데이터로더를 사용한 배치 테스트
def test_with_loader(model, loader, num_samples=3):
    model.eval()
    with torch.no_grad():
        for i, (images, masks) in enumerate(loader):
            if i >= num_samples:
                break
                
            images = images.to(device)
            outputs = model(images)
            
            # 첫 번째 이미지만 시각화
            visualize_sample(images[0], masks[0], outputs[0])

# 테스트 실행 옵션
test_option = "single"  # "single" 또는 "loader"로 변경 가능

if test_option == "single":
    # 단일 이미지 테스트 (임의의 테스트 이미지 경로 지정)
    test_image_path = test_dataset.image_files[10]  # 10번째 이미지 선택
    print(f"Testing image: {test_image_path}")
    pred_mask = test_single_image(model, test_image_path, test_transform)
    
elif test_option == "loader":
    # 데이터로더를 사용한 배치 테스트
    print("Testing with data loader...")
    test_with_loader(model, test_loader, num_samples=3)
else:
    print("Invalid test option")

In [ ]:
import os
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as models

from PIL import Image
# 데이터셋 클래스
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask

# SCNN 모델 정의
class SCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        backbone = models.resnet34(pretrained=True)
        self.encoder1 = nn.Sequential(*list(backbone.children())[:5])
        self.encoder2 = backbone.layer2
        self.encoder3 = backbone.layer3
        self.encoder4 = backbone.layer4

        self.scnn = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=(1,9), padding=(0,4)),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=(9,1), padding=(4,0)),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        
        x = self.scnn(e4)
        x = self.decoder(x)
        return x

# 이미지 시각화 함수
def visualize_results(original_img, pred_mask, gt_mask=None):
    plt.figure(figsize=(18, 6))
    
    # 원본 이미지
    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title("Original Image")
    
    # GT 마스크 (있을 경우)
    if gt_mask is not None:
        gt_overlay = original_img.copy()
        gt_overlay[gt_mask == 1] = [1, 0, 0]  # 빨간색
        plt.subplot(1, 3, 2)
        plt.imshow(gt_overlay)
        plt.title("GT Overlay")
    
    # 예측 결과
    pred_overlay = original_img.copy()
    pred_overlay[pred_mask == 1] = [0, 1, 0]  # 초록색
    plt.subplot(1, 3, 3 if gt_mask is not None else 2)
    plt.imshow(pred_overlay)
    plt.title("Prediction")
    
    plt.tight_layout()
    plt.show()

# 단일 이미지 테스트 함수
def test_single_image(model, image_path, transform, label_path=None):
    # 이미지 로드 및 전처리
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to load image: {image_path}")
    original_img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_h, original_w = original_img.shape[:2]
    
    # 변환 적용
    transformed = transform(image=original_img, mask=np.zeros((original_h, original_w)))
    image_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # 예측 수행
    with torch.no_grad():
        output = model(image_tensor)
        pred_mask = F.interpolate(output, size=(original_h, original_w), 
                                mode='bilinear', align_corners=True)
        pred_mask = torch.argmax(pred_mask, dim=1).squeeze().cpu().numpy()
    
    # GT 마스크 로드 (있을 경우)
    gt_mask = None
    if label_path:
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
            gt_mask = np.zeros((original_h, original_w), dtype=np.uint8)
            for lane in label_data.get('geometry', []):
                if isinstance(lane, list) and len(lane) > 1:
                    points = np.array(lane, dtype=np.int32)
                    cv2.polylines(gt_mask, [points], isClosed=False, color=1, thickness=4)
            gt_mask = cv2.dilate(gt_mask, np.ones((3,3)), iterations=1)
        except Exception as e:
            print(f"Error loading label: {str(e)}")
    
    # 정규화 역변환
    img_np = image_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img_np = np.clip(img_np, 0, 1)
    # 결과 시각화 전에 gt_mask 크기 맞추기
    if gt_mask is not None:
        gt_mask = cv2.resize(gt_mask, (img_np.shape[1], img_np.shape[0]), interpolation=cv2.INTER_NEAREST)

    # 결과 시각화
    visualize_results(img_np, pred_mask, gt_mask)
    
    return pred_mask

# 메인 실행 코드
if __name__ == '__main__':
    # 디바이스 설정
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 데이터 변환
    test_transform = A.Compose([
        A.Resize(608, 1024),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    
    # 모델 로드
    model = SCNN(num_classes=2).to(device)
    model.load_state_dict(torch.load('/kaggle/input/scnn/other/default/1/SDLane_SCNN.pth', 
                                   map_location=device, weights_only=True))
    model.eval()
    
    # 테스트 이미지 경로
    test_image_path = '/kaggle/input/sdlane/SDLane/train/images/0932b1d66d21e2ce4de81086645ebd93955fb0c1/0011.jpg'
    test_label_path = test_image_path.replace('images', 'labels').replace('.jpg', '.json')
    
    # 단일 이미지 테스트 실행
    print(f"Testing image: {test_image_path}")
    pred_mask = test_single_image(model, test_image_path, test_transform, test_label_path)

In [ ]:

import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        # 비디오 폴더 탐색 로직
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        # 이미지 로드
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # JSON 주석 로드 (개선된 파싱)
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        # 마스크 생성 (개선된 버전)
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        # 후처리
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        # 변환 적용
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask

# 개선된 SCNN 모델
class SCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Backbone (ResNet34)
        backbone = models.resnet34(pretrained=True)
        self.encoder1 = nn.Sequential(*list(backbone.children())[:5])  # /2
        self.encoder2 = backbone.layer2  # /4
        self.encoder3 = backbone.layer3  # /8
        self.encoder4 = backbone.layer4  # /16

        # SCNN 모듈 (패딩 추가)
        self.scnn = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=(1,9), padding=(0,4)),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=(9,1), padding=(4,0)),
            nn.ReLU()
        )

        # 디코더 (크기 조정 보장)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, num_classes, kernel_size=1)  # 최종 출력
        )

    def forward(self, x):
        # 인코더
        e1 = self.encoder1(x)  # /2
        e2 = self.encoder2(e1) # /4
        e3 = self.encoder3(e2) # /8
        e4 = self.encoder4(e3) # /16
        
        # SCNN
        x = self.scnn(e4)
        
        # 디코더 (크기 자동 조정)
        x = self.decoder(x)
        return x
# 개선된 시각화 함수
def visualize_sample(image, mask, pred=None):
    plt.figure(figsize=(18, 6))
    
    # 1. 원본 이미지 복원 (정규화 해제)
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # 정규화 역변환
    img_np = np.clip(img_np, 0, 1)  # 픽셀값 [0,1]로 클리핑

    # 2. GT 마스크 처리 (차원 압축)
    mask_np = mask.cpu().numpy().squeeze()  # (H,W) 보장

    # 3. GT 오버레이 생성 (빨간색)
    overlay = img_np.copy()
    overlay[mask_np == 1] = [1, 0, 0]  # 차선 영역만 빨간색

    # 4. 시각화
    plt.subplot(1, 3, 1)
    plt.imshow(overlay)
    plt.title("Image + GT (Correct Overlay)")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title("GT Mask")

    if pred is not None:
        plt.subplot(1, 3, 3)
        # 예측 마스크 리사이즈 (원본 크기로 조정)
        pred_resized = F.interpolate(
            pred.unsqueeze(0).float(),  # 배치 차원 추가
            size=(img_np.shape[0], img_np.shape[1]),  # 원본 이미지 크기
            mode='bilinear',
            align_corners=True
        )
        pred_mask = torch.argmax(pred_resized, dim=1).squeeze().cpu().numpy()
        
        # 초록색 오버레이
        pred_overlay = img_np.copy()
        pred_overlay[pred_mask == 1] = [0, 1, 0]  # 차선 영역만 초록색
        plt.imshow(pred_overlay)
        plt.title("Prediction")

    plt.tight_layout()
    plt.show()
# 개선된 평가 메트릭
def calculate_metrics(outputs, targets):
    resized_targets = F.interpolate(targets.unsqueeze(1).float(), 
                                size=outputs.shape[2:], 
                                mode='nearest').squeeze(1).long()
    preds = torch.argmax(outputs, dim=1)
    
    # IoU 계산
    intersection = (preds & resized_targets).float().sum()
    union = (preds | resized_targets).float().sum()
    iou = (intersection + 1e-6) / (union + 1e-6)
    
    # Precision, Recall, F1 계산
    tp = ((preds == 1) & (resized_targets == 1)).float().sum()
    fp = ((preds == 1) & (resized_targets == 0)).float().sum()
    fn = ((preds == 0) & (resized_targets == 1)).float().sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    return iou, precision, recall, f1

# 개선된 데이터 증강
train_transform = A.Compose([
    A.Resize(608, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomShadow(shadow_roi=(0,0,1,0.5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.GridDropout(ratio=0.1, random_offset=True, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(),
    ToTensorV2()
])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------- 2. 모델 정의 및 로드 --------------------
model = SCNN(num_classes=2).to(device)
model.load_state_dict(torch.load('/kaggle/input/scnn/other/default/1/SDLane_SCNN.pth', map_location=device))
model.eval()

dataset = SDLaneDataset(
    image_dir='/kaggle/input/sdlane/SDLane/train/images',
    label_dir='/kaggle/input/sdlane/SDLane/train/labels',
    transform=train_transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform
    
# 데이터로더
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
# -------------------- 4. 평가 --------------------
val_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
total_samples = 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        iou, prec, rec, f1 = calculate_metrics(outputs, masks)

        batch_size = images.size(0)
        total_samples += batch_size
        val_metrics['iou'] += iou.item() * batch_size
        val_metrics['precision'] += prec.item() * batch_size
        val_metrics['recall'] += rec.item() * batch_size
        val_metrics['f1'] += f1.item() * batch_size

# 평균 메트릭 계산
val_metrics = {k: v / total_samples for k, v in val_metrics.items()}

print(f"\n[Best Model Evaluation]")
print(f"IoU: {val_metrics['iou']:.4f} | Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")

# -------------------- 5. 예측 시각화 --------------------
with torch.no_grad():
    val_images, val_masks = next(iter(val_loader))
    val_images, val_masks = val_images.to(device), val_masks.to(device)
    val_outputs = model(val_images)

    for i in range(min(3, len(val_images))):
        visualize_sample(val_images[i], val_masks[i], val_outputs[i])


In [ ]:

import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        # 비디오 폴더 탐색 로직
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        # 이미지 로드
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # JSON 주석 로드 (개선된 파싱)
        try:
            with open(label_path, 'r') as f:
                label_data = json.load(f)
        except Exception as e:
            raise ValueError(f"JSON load error {label_path}: {str(e)}")
        
        # 마스크 생성 (개선된 버전)
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane in lanes:
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(mask, [points], isClosed=False, color=1, thickness=4)
        
        # 후처리
        mask = cv2.dilate(mask, np.ones((3,3)), iterations=1)
        
        # 변환 적용
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()
        
        return image, mask

# 개선된 SCNN 모델
class DownsamplerBlock(nn.Module):
    def __init__(self, ninput, noutput):
        super().__init__()
        self.conv = nn.Conv2d(ninput, noutput - ninput, (3, 3), stride=2, padding=1, bias=False)
        self.pool = nn.MaxPool2d(2, stride=2)
        self.bn = nn.BatchNorm2d(noutput, eps=1e-3)

    def forward(self, input):
        output = torch.cat([self.conv(input), self.pool(input)], 1)
        output = self.bn(output)
        return F.relu(output)

class NonBottleneck1D(nn.Module):
    def __init__(self, chann, dropprob, dilated):
        super().__init__()
        # 패딩 값 조정
        self.conv3x1_1 = nn.Conv2d(chann, chann, (3, 1), stride=1, padding=(1, 0), bias=True)
        self.conv1x3_1 = nn.Conv2d(chann, chann, (1, 3), stride=1, padding=(0, 1), bias=True)
        self.bn1 = nn.BatchNorm2d(chann, eps=1e-03)
        
        # 팽창 컨볼루션 패딩 조정
        self.conv3x1_2 = nn.Conv2d(chann, chann, (3, 1), stride=1, 
                                  padding=(dilated, 0), dilation=(dilated, 1), bias=True)
        self.conv1x3_2 = nn.Conv2d(chann, chann, (1, 3), stride=1, 
                                  padding=(0, dilated), dilation=(1, dilated), bias=True)
        self.bn2 = nn.BatchNorm2d(chann, eps=1e-03)
        self.dropout = nn.Dropout2d(dropprob)

    def forward(self, input):
        output = self.conv3x1_1(input)
        output = F.relu(output)
        output = self.conv1x3_1(output)
        output = self.bn1(output)
        output = F.relu(output)

        output = self.conv3x1_2(output)
        output = F.relu(output)
        output = self.conv1x3_2(output)
        output = self.bn2(output)

        if self.dropout.p != 0:
            output = self.dropout(output)
            
        # Skip connection 전에 크기 확인 및 조정
        if output.shape != input.shape:
            output = F.interpolate(output, size=input.shape[2:], mode='bilinear', align_corners=True)
            
        return F.relu(output + input)

class ERFNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Encoder
        self.initial_block = DownsamplerBlock(3, 16)
        
        self.layers = nn.ModuleList()
        self.layers.append(DownsamplerBlock(16, 64))
        
        # 첫 번째 NonBottleneck 블록 (dilation=1)
        for _ in range(5):
            self.layers.append(NonBottleneck1D(64, 0.1, 1))
        
        self.layers.append(DownsamplerBlock(64, 128))
        
        # 팽창 컨볼루션 블록 (다양한 dilation rate)
        dilation_rates = [2, 4, 8, 16] * 2  # 2번 반복
        for rate in dilation_rates:
            self.layers.append(NonBottleneck1D(128, 0.1, rate))
        
        # Decoder
        self.decoder_layers = nn.ModuleList()
        
        # 첫 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(128, 64, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(64))
        self.decoder_layers.append(nn.ReLU())
        
        # 두 번째 업샘플링
        self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
        self.decoder_layers.append(nn.Conv2d(64, 16, kernel_size=3, padding=1))
        self.decoder_layers.append(nn.BatchNorm2d(16))
        self.decoder_layers.append(nn.ReLU())
        
        # 최종 출력 레이어
        self.output_conv = nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x = self.initial_block(x)
        for layer in self.layers:
            x = layer(x)
        
        # Decoder
        for layer in self.decoder_layers:
            x = layer(x)
        
        return self.output_conv(x)

# 시각화 함수 (기존과 동일)
def visualize_sample(image, mask, pred=None):
    plt.figure(figsize=(18, 6))
    
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img_np = np.clip(img_np, 0, 1)

    mask_np = mask.cpu().numpy().squeeze()

    overlay = img_np.copy()
    overlay[mask_np == 1] = [1, 0, 0]

    plt.subplot(1, 3, 1)
    plt.imshow(overlay)
    plt.title("Image + GT")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title("GT Mask")

    if pred is not None:
        plt.subplot(1, 3, 3)
        pred_resized = F.interpolate(
            pred.unsqueeze(0).float(),
            size=(img_np.shape[0], img_np.shape[1]),
            mode='bilinear',
            align_corners=True
        )
        pred_mask = torch.argmax(pred_resized, dim=1).squeeze().cpu().numpy()
        
        pred_overlay = img_np.copy()
        pred_overlay[pred_mask == 1] = [0, 1, 0]
        plt.imshow(pred_overlay)
        plt.title("Prediction")

    plt.tight_layout()
    plt.show()

# 평가 메트릭 (기존과 동일)
def calculate_metrics(outputs, targets):
    resized_targets = F.interpolate(targets.unsqueeze(1).float(), 
                                size=outputs.shape[2:], 
                                mode='nearest').squeeze(1).long()
    preds = torch.argmax(outputs, dim=1)
    
    intersection = (preds & resized_targets).float().sum()
    union = (preds | resized_targets).float().sum()
    iou = (intersection + 1e-6) / (union + 1e-6)
    
    tp = ((preds == 1) & (resized_targets == 1)).float().sum()
    fp = ((preds == 1) & (resized_targets == 0)).float().sum()
    fn = ((preds == 0) & (resized_targets == 1)).float().sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    return iou, precision, recall, f1

# 데이터 증강 (기존과 동일)
train_transform = A.Compose([
    A.Resize(608, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomShadow(shadow_roi=(0,0,1,0.5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.GridDropout(ratio=0.1, random_offset=True, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------- 2. 모델 정의 및 로드 --------------------
model = ERFNet(num_classes=2).to(device)
model.load_state_dict(torch.load('/kaggle/input/erfnet/other/default/1/SDLane_ERFNet.pth', map_location=device))
model.eval()

dataset = SDLaneDataset(
    image_dir='/kaggle/input/sdlane/SDLane/train/images',
    label_dir='/kaggle/input/sdlane/SDLane/train/labels',
    transform=train_transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform
    
# 데이터로더
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
# -------------------- 4. 평가 --------------------
val_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
total_samples = 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        iou, prec, rec, f1 = calculate_metrics(outputs, masks)

        batch_size = images.size(0)
        total_samples += batch_size
        val_metrics['iou'] += iou.item() * batch_size
        val_metrics['precision'] += prec.item() * batch_size
        val_metrics['recall'] += rec.item() * batch_size
        val_metrics['f1'] += f1.item() * batch_size

# 평균 메트릭 계산
val_metrics = {k: v / total_samples for k, v in val_metrics.items()}

print(f"\n[Best Model Evaluation]")
print(f"IoU: {val_metrics['iou']:.4f} | Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")

# -------------------- 5. 예측 시각화 --------------------
with torch.no_grad():
    val_images, val_masks = next(iter(val_loader))
    val_images, val_masks = val_images.to(device), val_masks.to(device)
    val_outputs = model(val_images)

    for i in range(min(3, len(val_images))):
        visualize_sample(val_images[i], val_masks[i], val_outputs[i])


In [ ]:
import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import GradScaler, autocast
from torchvision.models import vgg16

# 데이터셋 클래스 (LaneNet용으로 수정)
class SDLaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = []
        self.label_files = []
        
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise ValueError("Image or label directory does not exist")
        
        for video_folder in sorted(os.listdir(image_dir)):
            video_img_path = os.path.join(image_dir, video_folder)
            video_label_path = os.path.join(label_dir, video_folder)
            
            if os.path.isdir(video_img_path) and os.path.isdir(video_label_path):
                for file in sorted(os.listdir(video_img_path)):
                    if file.endswith(".jpg"):
                        img_path = os.path.join(video_img_path, file)
                        label_path = os.path.join(video_label_path, file.replace(".jpg", ".json"))
                        
                        if os.path.exists(label_path):
                            self.image_files.append(img_path)
                            self.label_files.append(label_path)

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_files[idx]
        
        # 이미지 로드
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # JSON 주석 로드
        with open(label_path, 'r') as f:
            label_data = json.load(f)
        
        # 마스크 생성
        binary_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        instance_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        lanes = label_data.get('geometry', [])
        
        for lane_idx, lane in enumerate(lanes, start=1):
            if isinstance(lane, list) and len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(binary_mask, [points], isClosed=False, color=1, thickness=4)
                cv2.polylines(instance_mask, [points], isClosed=False, color=lane_idx, thickness=4)
        
        # 후처리
        kernel = np.ones((3, 3), np.uint8)
        binary_mask = cv2.dilate(binary_mask, np.ones((3,3)), iterations=1)
        instance_mask = cv2.dilate(instance_mask, np.ones((3,3)), iterations=1)
        # 변환 적용 (numpy 배열로 전달)
        if self.transform:
            # Albumentations에 전달하기 전에 반드시 numpy 배열인지 확인
            transformed = self.transform(
                image=image.astype(np.uint8),  # 이미지는 uint8 형식으로
                masks=[
                    binary_mask.astype(np.uint8),  # 바이너리 마스크
                    instance_mask.astype(np.uint8)  # 인스턴스 마스크
                ]
            )
            image = transformed['image']
            binary_mask = transformed['masks'][0].long()
            instance_mask = transformed['masks'][1].long()
        else:
            # 변환 없을 경우 torch.Tensor로 변환
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            binary_mask = torch.from_numpy(binary_mask).long()
            instance_mask = torch.from_numpy(instance_mask).long()
        
        return image, {'binary': binary_mask, 'instance': instance_mask}
# LaneNet 모델 구현
class LaneNet(nn.Module):
    def __init__(self, num_classes=2, embedding_dim=4):
        super().__init__()
        # Backbone Network (VGG16 기반)
        backbone = vgg16(weights='DEFAULT').features
        self.encoder1 = backbone[:4]    # /1
        self.encoder2 = backbone[4:9]   # /2
        self.encoder3 = backbone[9:16]  # /4
        self.encoder4 = backbone[16:23] # /8
        self.encoder5 = backbone[23:30] # /16

        # Binary Segmentation Decoder
        self.binary_decoder = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(16, num_classes, kernel_size=1)
        )

        # Embedding Decoder
        self.embedding_decoder = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(16, embedding_dim, kernel_size=1)
        )

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)   # /1
        e2 = self.encoder2(e1)  # /2
        e3 = self.encoder3(e2)  # /4
        e4 = self.encoder4(e3)  # /8
        e5 = self.encoder5(e4)  # /16

        # Decoders
        binary_seg = self.binary_decoder(e5)
        embedding = self.embedding_decoder(e5)

        return binary_seg, embedding

# Discriminative Loss (인스턴스 구분을 위한 손실)
class DiscriminativeLoss(nn.Module):
    def __init__(self, delta_var=0.5, delta_dist=1.5):
        super().__init__()
        self.delta_var = delta_var
        self.delta_dist = delta_dist
        
    def forward(self, embedding, instance_gt):
        batch_size, embed_dim, H, W = embedding.shape
        loss = 0.0
        
        for b in range(batch_size):
            emb = embedding[b].view(embed_dim, -1)  # [embed_dim, H*W]
            inst = instance_gt[b].view(-1)          # [H*W]
            
            unique_instances = torch.unique(inst)
            # 배경(0) 제외하고 실제 차선 인스턴스만 필터링
            unique_instances = unique_instances[unique_instances != 0]
            num_instances = len(unique_instances)
            
            if num_instances == 0:
                continue
                
            # 인스턴스별 중심 계산
            centers = []
            valid_instances = []
            for inst_id in unique_instances:
                mask = (inst == inst_id)
                if mask.sum() == 0:  # 픽셀이 없는 인스턴스는 건너뜀
                    continue
                center = emb[:, mask].mean(dim=1)
                centers.append(center)
                valid_instances.append(inst_id)
            
            if len(centers) == 0:  # 유효한 인스턴스가 없는 경우
                continue
                
            centers = torch.stack(centers)  # [num_valid_instances, embed_dim]
            
            # Variance term (동일 인스턴스 내 거리 최소화)
            var_loss = 0.0
            for i, inst_id in enumerate(valid_instances):
                mask = (inst == inst_id)
                diff = torch.norm(emb[:, mask] - centers[i].view(-1, 1), dim=0)
                var_loss += torch.mean(F.relu(diff - self.delta_var)**2)
            
            if num_instances > 0:
                var_loss /= num_instances
            
            # Distance term (다른 인스턴스 간 거리 최대화)
            dist_loss = 0.0
            if len(centers) > 1:  # 2개 이상의 인스턴스가 있을 때만 계산
                center_pairs = torch.combinations(torch.arange(len(centers)), 2)
                for i, j in center_pairs:
                    diff = torch.norm(centers[i] - centers[j])
                    dist_loss += torch.mean(F.relu(self.delta_dist - diff)**2)
                dist_loss /= len(center_pairs)
            
            # Regularization term (중심을 원점으로 끌어당김)
            reg_loss = torch.mean(torch.norm(centers, dim=1))
            
            loss += var_loss + dist_loss + 0.001 * reg_loss
        
        return loss / batch_size if batch_size > 0 else 0.0
# LaneNet용 손실 함수
class LaneNetLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.binary_loss = nn.CrossEntropyLoss()
        self.embedding_loss = DiscriminativeLoss()
        
    def forward(self, outputs, targets):
        binary_pred, embedding_pred = outputs
        binary_gt, instance_gt = targets
        
        # Binary segmentation loss
        loss_bin = self.binary_loss(binary_pred, binary_gt)
        
        # Embedding loss
        loss_emb = self.embedding_loss(embedding_pred, instance_gt)
        
        return loss_bin + 0.1 * loss_emb  # 가중치 조정

# 데이터 증강 (멀티 마스크 지원)
train_transform = A.Compose([
    A.Resize(608, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomShadow(shadow_roi=(0,0,1,0.5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.GridDropout(ratio=0.1, random_offset=True, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={'masks': 'mask'})

val_transform = A.Compose([
    A.Resize(608, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={
    'mask0': 'mask',
    'mask1': 'mask'
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------- 2. 모델 정의 및 로드 --------------------
model = LaneNet(num_classes=2, embedding_dim=4).to(device)
model.load_state_dict(torch.load('/kaggle/input/lanenet/other/default/1/best_lanenet (2).pth', map_location=device))
model.eval()

dataset = SDLaneDataset(
    image_dir='/kaggle/input/sdlane/SDLane/train/images',
    label_dir='/kaggle/input/sdlane/SDLane/train/labels',
    transform=train_transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform
    
# 데이터로더
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
# -------------------- 4. 평가 --------------------
def calculate_metrics(outputs, targets):
    binary_pred, _ = outputs
    binary_gt = targets['binary']  # Get binary mask from dictionary
    
    # Resize predictions to match target size if needed
    if binary_pred.shape[2:] != binary_gt.shape[1:]:
        binary_pred = F.interpolate(binary_pred, size=binary_gt.shape[1:], 
                                  mode='bilinear', align_corners=True)
    
    preds = torch.argmax(binary_pred, dim=1)
    
    # IoU calculation
    intersection = (preds & binary_gt).float().sum()
    union = (preds | binary_gt).float().sum()
    iou = (intersection + 1e-6) / (union + 1e-6)
    
    # Precision, Recall, F1 calculation
    tp = ((preds == 1) & (binary_gt == 1)).float().sum()
    fp = ((preds == 1) & (binary_gt == 0)).float().sum()
    fn = ((preds == 0) & (binary_gt == 1)).float().sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    return iou, precision, recall, f1

val_metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
total_samples = 0

model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        # Move data to device
        images = images.to(device)
        masks = {k: v.to(device) for k, v in masks.items()}
        
        # Forward pass
        outputs = model(images)
        
        # Calculate metrics
        iou, prec, rec, f1 = calculate_metrics(outputs, masks)

        # Accumulate metrics
        batch_size = images.size(0)
        total_samples += batch_size
        val_metrics['iou'] += iou.item() * batch_size
        val_metrics['precision'] += prec.item() * batch_size
        val_metrics['recall'] += rec.item() * batch_size
        val_metrics['f1'] += f1.item() * batch_size

# Calculate average metrics
val_metrics = {k: v / total_samples for k, v in val_metrics.items()}
# -------------------- 5. 예측 시각화 --------------------
def visualize_sample(image, masks, outputs):
    binary_pred, embedding_pred = outputs
    binary_gt = masks['binary']  # Get binary mask from dictionary
    instance_gt = masks['instance']  # Get instance mask from dictionary
    
    # Process prediction
    binary_pred = torch.argmax(binary_pred, dim=0).cpu().numpy()
    
    # Prepare original image
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img_np = np.clip(img_np, 0, 1)
    
    # Create overlays
    gt_overlay = img_np.copy()
    gt_overlay[binary_gt.cpu().numpy() == 1] = [1, 0, 0]  # Red for GT
    
    pred_overlay = img_np.copy()
    pred_overlay[binary_pred == 1] = [0, 1, 0]  # Green for prediction
    
    plt.figure(figsize=(18, 6))
    
    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title("Original Image")
    
    # Ground truth
    plt.subplot(1, 3, 2)
    plt.imshow(gt_overlay)
    plt.title("Ground Truth")
    
    # Prediction
    plt.subplot(1, 3, 3)
    plt.imshow(pred_overlay)
    plt.title("Prediction")
    
    plt.tight_layout()
    plt.show()

# Evaluation results
print(f"\n[Best Model Evaluation]")
print(f"IoU: {val_metrics['iou']:.4f} | Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f}")

# Visualization
with torch.no_grad():
    val_images, val_masks = next(iter(val_loader))
    val_images = val_images.to(device)
    val_masks = {k: v.to(device) for k, v in val_masks.items()}
    val_outputs = model(val_images)

    # Get first 3 samples
    for i in range(min(3, len(val_images))):
        # Extract individual sample
        sample_image = val_images[i]
        sample_masks = {k: v[i] for k, v in val_masks.items()}
        sample_outputs = (val_outputs[0][i], val_outputs[1][i])
        
        visualize_sample(sample_image, sample_masks, sample_outputs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 각 모델의 지표별 5회 측정값
metrics = {
    'SCNN': {
        'IoU': [0.4902, 0.4890, 0.4921, 0.4895, 0.4967],
        'Precision': [0.4925, 0.4913, 0.4944, 0.4918, 0.4991],
        'Recall': [0.9893, 0.9896, 0.9896, 0.9892, 0.9897],
        'F1': [0.6550, 0.6541, 0.6571, 0.6544, 0.6613],
    },
    'ERFNet': {
        'IoU': [0.3219, 0.3224, 0.3229, 0.3260, 0.3246],
        'Precision': [0.3232, 0.3236, 0.3241, 0.3272, 0.3259],
        'Recall': [0.9865, 0.9863, 0.9864, 0.9871, 0.9862],
        'F1': [0.4853, 0.4855, 0.4862, 0.4899, 0.4880],
    },
    'LaneNet': {
        'IoU': [0.4443, 0.4451, 0.4460, 0.4464, 0.4424],
        'Precision': [0.7376, 0.7393, 0.7403, 0.7391, 0.7388],
        'Recall': [0.5276, 0.5278, 0.5288, 0.5297, 0.5244],
        'F1': [0.6141, 0.6146, 0.6155, 0.6159, 0.6121],
    },
}

# 평균 계산
avg_metrics = {model: {k: np.mean(v) for k, v in metric.items()} for model, metric in metrics.items()}

# 그래프 시각화
categories = ['IoU', 'Precision', 'Recall', 'F1']
x = np.arange(len(categories))
width = 0.25

scnn_vals = [avg_metrics['SCNN'][c] for c in categories]
erfnet_vals = [avg_metrics['ERFNet'][c] for c in categories]
lanenet_vals = [avg_metrics['LaneNet'][c] for c in categories]

plt.figure(figsize=(10, 6))
plt.bar(x - width, scnn_vals, width, label='SCNN')
plt.bar(x, erfnet_vals, width, label='ERFNet')
plt.bar(x + width, lanenet_vals, width, label='LaneNet')

plt.ylabel('Metric Score')
plt.title('Comparison of SCNN, ERFNet, LaneNet (Average over 5 Runs)')
plt.xticks(x, categories)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 모델별 평가지표 (5회 평균)
scnn_metrics = {
    "IoU": [0.4902, 0.4890, 0.4921, 0.4895, 0.4967],
    "Precision": [0.4925, 0.4913, 0.4944, 0.4918, 0.4991],
    "Recall": [0.9893, 0.9896, 0.9896, 0.9892, 0.9897],
    "F1": [0.6550, 0.6541, 0.6571, 0.6544, 0.6613]
}

erfnet_metrics = {
    "IoU": [0.3219, 0.3224, 0.3229, 0.3260, 0.3246],
    "Precision": [0.3232, 0.3236, 0.3241, 0.3272, 0.3259],
    "Recall": [0.9865, 0.9863, 0.9864, 0.9871, 0.9862],
    "F1": [0.4853, 0.4855, 0.4862, 0.4899, 0.4880]
}

lanenet_metrics = {
    "IoU": [0.4443, 0.4451, 0.4460, 0.4464, 0.4424],
    "Precision": [0.7376, 0.7393, 0.7403, 0.7391, 0.7388],
    "Recall": [0.5276, 0.5278, 0.5288, 0.5297, 0.5244],
    "F1": [0.6141, 0.6146, 0.6155, 0.6159, 0.6121]
}

# 평균 계산 함수
def average(metric_list):
    return sum(metric_list) / len(metric_list)

# 지표별 평균값
metrics = ["IoU", "Precision", "Recall", "F1"]
scnn_avg = [average(scnn_metrics[m]) for m in metrics]
erfnet_avg = [average(erfnet_metrics[m]) for m in metrics]
lanenet_avg = [average(lanenet_metrics[m]) for m in metrics]

# 각 지표별 그래프 출력
models = ["SCNN", "ERFNet", "LaneNet"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # 파란색, 주황색, 초록색

for i, metric in enumerate(metrics):
    values = [scnn_avg[i], erfnet_avg[i], lanenet_avg[i]]
    
    plt.figure(figsize=(6, 4))
    plt.bar(models, values, color=colors)
    plt.title(f"{metric} Comparison")
    plt.ylabel(metric)
    plt.ylim(0, 1.1 if metric == "Recall" else 1.0)
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.transforms import functional as TF

class SCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Backbone (ResNet34)
        backbone = models.resnet34(weights=None)
        self.encoder1 = nn.Sequential(*list(backbone.children())[:5])  # /2
        self.encoder2 = backbone.layer2  # /4
        self.encoder3 = backbone.layer3  # /8
        self.encoder4 = backbone.layer4  # /16

        # SCNN 모듈 (패딩 추가)
        self.scnn = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=(1,9), padding=(0,4)),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=(9,1), padding=(4,0)),
            nn.ReLU()
        )

        # 디코더 (크기 조정 보장)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, num_classes, kernel_size=1)  # 최종 출력
        )

    def forward(self, x):
        # 인코더
        e1 = self.encoder1(x)  # /2
        e2 = self.encoder2(e1) # /4
        e3 = self.encoder3(e2) # /8
        e4 = self.encoder4(e3) # /16
        
        # SCNN
        x = self.scnn(e4)
        
        # 디코더 (크기 자동 조정)
        x = self.decoder(x)
        return x

# 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 모델 인스턴스 생성 및 가중치 로드
model = SCNN(num_classes=2).to(device)

# 모델 가중치 로드 (같은 장치로 이동)
state_dict = torch.load('/kaggle/input/scnn/other/default/1/SDLane_SCNN.pth', map_location=device)
model.load_state_dict(state_dict)
model.eval()

# 더미 입력 생성 (모델 입력 크기에 맞게, 같은 장치 사용)
dummy_input = torch.randn(1, 3, 608, 1024, device=device)

# ONNX로 변환
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/scnn.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        'input': {0: 'batch_size'},  # 동적 배치 크기
        'output': {0: 'batch_size'}
    },
    opset_version=11,
    verbose=True
)

print("ONNX export completed successfully!")

In [ ]:
# TensorRT 설치 (JetPack을 사용하는 경우 이미 설치되어 있음)
pip install tensorrt

# Python 바인딩 설치
pip install nvidia-pyindex
pip install nvidia-tensorrt

# ONNX 변환을 위한 패키지
pip install onnx onnxruntime

In [ ]:
import tensorrt as trt
import os

def build_engine(onnx_file_path, engine_file_path):
    TRT_LOGGER = trt.Logger(trt.Logger.VERBOSE)
    builder = trt.Builder(TRT_LOGGER)
    network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(network_flags)

    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_file_path, 'rb') as model:
        if not parser.parse(model.read()):
            print("ONNX parsing failed:")
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
        print("✓ ONNX parsed successfully")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    # ★ Optimization Profile 설정
    input_name = network.get_input(0).name
    profile = builder.create_optimization_profile()
    profile.set_shape(
        input_name,
        min=(1, 3, 608, 1024),
        opt=(1, 3, 608, 1024),
        max=(1, 3, 608, 1024)
    )
    config.add_optimization_profile(profile)

    print("Building TensorRT engine...")
    try:
        engine = builder.build_serialized_network(network, config)
        if engine is None:
            raise RuntimeError("Engine build returned None")
        with open(engine_file_path, 'wb') as f:
            f.write(engine)
        print(f"✓ Engine saved to {engine_file_path}")
        return engine
    except Exception as e:
        print(f"Engine build failed: {str(e)}")
        return None

# Usage
onnx_path = "/kaggle/input/onxx/other/default/1/scnn.onnx"
engine_path = "/kaggle/working/scnn.trt"

if not os.path.exists(onnx_path):
    raise FileNotFoundError(f"ONNX file not found at {onnx_path}")

print(f"Building engine from {onnx_path}...")
engine = build_engine(onnx_path, engine_path)

if engine:
    print("Engine built successfully!")
else:
    print("Failed to build engine")

In [ ]:
pip install onnx


In [ ]:
import onnx
model = onnx.load("/kaggle/input/onxx/other/default/1/scnn.onnx")
for i in model.graph.input:
    print(f"Input name: {i.name}")
    for dim in i.type.tensor_type.shape.dim:
        print(f" - dim: {dim.dim_value if dim.HasField('dim_value') else '?'}")
